In [1]:
!free -h
!nvidia-smi --query-gpu=name,memory.total --format=csv

               total        used        free      shared  buff/cache   available
Mem:            52Gi       1.4Gi        48Gi       2.2Mi       3.8Gi        51Gi
Swap:             0B          0B          0B
name, memory.total [MiB]
NVIDIA L4, 23034 MiB


install and record versions

In [13]:
!pip install -q vllm==0.30.0 openai pillow requests

import os, io, sys, json, time, base64, subprocess, requests
import vllm, torch, transformers
from PIL import Image
from google.colab import userdata

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"

VERSIONS = {
    "python": sys.version.split()[0],
    "vllm": vllm.__version__,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "model": MODEL,
}
print(json.dumps(VERSIONS, indent=2))

{
  "python": "3.13.15",
  "vllm": "0.30.0",
  "torch": "2.13.0+cu130",
  "cuda": "13.0",
  "transformers": "5.18.0",
  "gpu": "NVIDIA L4",
  "model": "Qwen/Qwen2.5-VL-3B-Instruct"
}


start the vLLM server

In [14]:
SERVER_ARGS = ["--max-model-len", "16384", "--gpu-memory-utilization", "0.85", "--port", "8000"]

log = open("/content/server_gate1.log", "w")
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL, *SERVER_ARGS],
    stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())

t0 = time.time()
while True:
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print(f"Server up in {time.time() - t0:.0f}s"); break
    except requests.exceptions.RequestException:
        pass
    if server.poll() is not None or time.time() - t0 > 900:
        print("Server failed. Last log lines:")
        print(open("/content/server_gate1.log").read()[-3000:]); break
    time.sleep(5)

Server up in 0s


prepare the stop sign image

In [15]:
from vllm.assets.image import ImageAsset

img = ImageAsset("stop_sign").pil_image.convert("RGB")
buf = io.BytesIO()
img.save(buf, format="JPEG", quality=95)
JPEG_BYTES = buf.getvalue()

DATA_URL = "data:image/jpeg;base64," + base64.b64encode(JPEG_BYTES).decode()
SENT_IMG = Image.open(io.BytesIO(JPEG_BYTES)).convert("RGB")   # exactly what vLLM will decode
W, H = SENT_IMG.size
print(f"width {W}, height {H}, JPEG {len(JPEG_BYTES)/1024:.0f} KB, base64 {len(DATA_URL)/1024:.0f} KB")

width 1300, height 876, JPEG 248 KB, base64 330 KB


gate test #1

In [5]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
PROMPT = "Describe this image in one sentence."

def ask(messages):
    """Send one non-streaming chat request to vLLM.

    Use temperature=0 and max_tokens=60 (same as the original test #1).
    Returns (prompt_tokens, answer_text).
    """
    r = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0,
        max_tokens=60,
    )
    return r.usage.prompt_tokens, r.choices[0].message.content

img_messages = [{
    "role": "user",
    "content": [
        {"type": "image_url", "image_url": {"url": DATA_URL}},
        {"type": "text", "text": PROMPT},
    ],
}]

txt_messages = [{
    "role": "user",
    "content": PROMPT,
}]

img_prompt_tokens, answer = ask(img_messages)
txt_prompt_tokens, _ = ask(txt_messages)

measured = img_prompt_tokens - txt_prompt_tokens
expected = round(H / 28) * round(W / 28)

print("answer:", answer)
print(f"prompt tokens with image: {img_prompt_tokens}, text only: {txt_prompt_tokens}")
print(f"measured image tokens: {measured}, 28x28 rule: {expected}, gap: {measured - expected}")

answer: A stop sign is in front of a red and gold gate with Chinese characters on it.
prompt tokens with image: 1454, text only: 26
measured image tokens: 1428, 28x28 rule: 1426, gap: 2


settle the marker question

In [6]:
from transformers import AutoProcessor

proc = AutoProcessor.from_pretrained(MODEL)

def count_tokens_hf(pil_img, prompt):
    """Build the exact prompt Qwen sees for (image + prompt) and count tokens by type.

    Returns a dict: total, vision_start, image_pad, vision_end, grid.
    """
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": prompt}]}]

    text = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    print(text)

    inputs = proc(text=[text], images=[pil_img], return_tensors="pt")
    ids = inputs["input_ids"][0].tolist()

    tid = proc.tokenizer.convert_tokens_to_ids
    return {
        "total": len(ids),
        "vision_start": ids.count(tid("<|vision_start|>")),
        "image_pad": ids.count(tid("<|image_pad|>")),
        "vision_end": ids.count(tid("<|vision_end|>")),
        "grid": inputs["image_grid_thw"][0].tolist(),
    }

hf = count_tokens_hf(SENT_IMG, PROMPT)
print(hf)
print("HF total == vLLM prompt_tokens:   ", hf["total"] == img_prompt_tokens)
print("image_pad == 28x28 rule:          ", hf["image_pad"] == expected)
print("start + pads + end == measured:   ", hf["vision_start"] + hf["image_pad"] + hf["vision_end"] == measured)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Describe this image in one sentence.<|im_end|>
<|im_start|>assistant

{'total': 1454, 'vision_start': 1, 'image_pad': 1426, 'vision_end': 1, 'grid': [1, 62, 92]}
HF total == vLLM prompt_tokens:    True
image_pad == 28x28 rule:           True
start + pads + end == measured:    True


save the result

In [7]:
from google.colab import drive
import datetime, shutil

drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/visionabr_results/phase_minus1"
os.makedirs(DEST, exist_ok=True)

result = {
    "test": "gate1_model_runs_and_token_count",
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "versions": VERSIONS,
    "config": {
        "server_args": SERVER_ARGS,
        "image": "vllm ImageAsset stop_sign, JPEG quality 95, base64 data URL",
        "image_size_wh": [W, H],
        "prompt": PROMPT,
        "temperature": 0,
        "max_tokens": 60,
        "max_pixels": "default",
        "streaming": False,
    },
    "raw": {
        "answer": answer,
        "prompt_tokens_with_image": img_prompt_tokens,
        "prompt_tokens_text_only": txt_prompt_tokens,
        "hf_processor_counts": hf,
    },
    "summary": {
        "measured_image_tokens": measured,
        "rule_28x28": expected,
        "gap": measured - expected,
        "markers_verified": hf["vision_start"] == 1 and hf["vision_end"] == 1
                            and hf["vision_start"] + hf["image_pad"] + hf["vision_end"] == measured,
    },
}

stamp = datetime.date.today().isoformat()
path = f"{DEST}/{stamp}_gate1_tokens_stopsign.json"
json.dump(result, open(path, "w"), indent=2)
print("saved:", path)

shutil.copy("/content/server_gate1.log", f"{DEST}/{stamp}_gate1_server.log")
print("saved:", f"{DEST}/{stamp}_gate1_server.log")

Mounted at /content/drive
saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate1_tokens_stopsign.json
saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate1_server.log


gate test #2

In [8]:
import math

SIZES = {"full": None, "~1024": 802816, "~512": 401408, "~256": 200704}

def predicted_image_tokens(w, h, max_pixels=None, f=28):
    """Qwen2.5-VL resize rule: keep shape, round sides to multiples of 28,
    and if over max_pixels, scale down and round DOWN. Returns (tokens incl. 2 markers, (h, w))."""
    h_bar, w_bar = round(h / f) * f, round(w / f) * f
    if max_pixels is not None and h_bar * w_bar > max_pixels:
        beta = math.sqrt(h * w / max_pixels)
        h_bar = math.floor(h / beta / f) * f
        w_bar = math.floor(w / beta / f) * f
    return (h_bar // f) * (w_bar // f) + 2, (h_bar, w_bar)

def ask_sized(max_px):
    """Same image request as gate #1, with an optional per-request max_pixels."""
    extra = {"mm_processor_kwargs": {"max_pixels": max_px}} if max_px else None
    r = client.chat.completions.create(
        model=MODEL, messages=img_messages, temperature=0, max_tokens=60, extra_body=extra)
    return r.usage.prompt_tokens, r.choices[0].message.content

gate2_rows = []
for name, px in SIZES.items():
    pt, ans = ask_sized(px)
    measured_img = pt - txt_prompt_tokens
    pred, (rh, rw) = predicted_image_tokens(W, H, px)
    gate2_rows.append({
        "size": name, "max_pixels": px, "prompt_tokens": pt,
        "image_tokens_incl_markers": measured_img, "predicted": pred,
        "resized_wh": [rw, rh], "answer": ans,
    })
    status = "OK" if measured_img == pred else "MISMATCH"
    print(f"{name:>6} | max_pixels {str(px):>7} | resized {rw}x{rh} | "
          f"image tokens {measured_img:>4} (rule {pred:>4}) {status} | {ans[:55]}")

  full | max_pixels    None | resized 1288x868 | image tokens 1428 (rule 1428) OK | A stop sign is in front of a red and gold gate with Chi
 ~1024 | max_pixels  802816 | resized 1064x728 | image tokens  990 (rule  990) OK | A stop sign is in front of a Chinese style building.
  ~512 | max_pixels  401408 | resized 756x504 | image tokens  488 (rule  488) OK | A stop sign is in front of a Chinese style archway.
  ~256 | max_pixels  200704 | resized 532x364 | image tokens  249 (rule  249) OK | A stop sign is in front of a Chinese restaurant.


save gate #2

In [9]:
gate2 = {
    "test": "gate2_per_request_max_pixels",
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "versions": VERSIONS,
    "config": {
        "server_args": SERVER_ARGS,
        "image": "vllm ImageAsset stop_sign, JPEG quality 95, base64 data URL (same image every request)",
        "image_size_wh": [W, H],
        "prompt": PROMPT, "temperature": 0, "max_tokens": 60, "streaming": False,
        "text_only_prompt_tokens": txt_prompt_tokens,
        "server_restarts_between_sizes": 0,
    },
    "raw": gate2_rows,
    "summary": {
        "image_tokens": {r["size"]: r["image_tokens_incl_markers"] for r in gate2_rows},
        "all_match_resize_rule": all(r["image_tokens_incl_markers"] == r["predicted"] for r in gate2_rows),
    },
}
path2 = f"{DEST}/{stamp}_gate2_max_pixels_stopsign.json"
json.dump(gate2, open(path2, "w"), indent=2)
print("saved:", path2)

saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate2_max_pixels_stopsign.json


gate test #3 pre-req work

restart the server with prefix caching off

In [16]:
server.terminate(); server.wait(); time.sleep(10)

SERVER_ARGS_G3 = SERVER_ARGS + ["--no-enable-prefix-caching"]
log3 = open("/content/server_gate3.log", "w")
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL, *SERVER_ARGS_G3],
    stdout=log3, stderr=subprocess.STDOUT, env=os.environ.copy())

t0 = time.time()
while True:
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print(f"Server up in {time.time() - t0:.0f}s"); break
    except requests.exceptions.RequestException:
        pass
    if server.poll() is not None or time.time() - t0 > 900:
        print("Server failed. Last log lines:")
        print(open("/content/server_gate3.log").read()[-3000:]); break
    time.sleep(5)

Server up in 0s


gate #3 and image share

In [11]:
import asyncio, statistics, itertools
import numpy as np
from openai import AsyncOpenAI

aclient = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key="none")
rng = np.random.default_rng(20261011)
BASE_ARR = np.array(img)
TEXT_IDS = itertools.count(100000)   # 6-digit ids, so every text prefix has the same token length

def unique_image_url():
    """Stop sign with a random 8x8 corner patch, so no two requests share an image."""
    arr = BASE_ARR.copy()
    arr[:8, :8] = rng.integers(0, 256, size=(8, 8, 3), dtype=np.uint8)
    b = io.BytesIO(); Image.fromarray(arr).save(b, format="JPEG", quality=95)
    return "data:image/jpeg;base64," + base64.b64encode(b.getvalue()).decode()

def img_factory():
    return [{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": unique_image_url()}},
        {"type": "text", "text": PROMPT}]}]

def text_factory(n_words):
    def make():
        return [{"role": "user", "content": f"[{next(TEXT_IDS)}] " + "word " * n_words + PROMPT}]
    return make

async def timed_request(messages, mm_kwargs=None):
    """Streaming request. TTFT = time until the first chunk with text."""
    extra = {"mm_processor_kwargs": mm_kwargs} if mm_kwargs else None
    t0 = time.perf_counter()
    stream = await aclient.chat.completions.create(
        model=MODEL, messages=messages, temperature=0, max_tokens=16,
        stream=True, stream_options={"include_usage": True}, extra_body=extra)
    ttft, ptoks = None, None
    async for chunk in stream:
        if ttft is None and chunk.choices and chunk.choices[0].delta.content:
            ttft = (time.perf_counter() - t0) * 1000
        if chunk.usage:
            ptoks = chunk.usage.prompt_tokens
    return {"ttft_ms": ttft, "prompt_tokens": ptoks}

async def run(factory, n, conc, mm_kwargs=None):
    """Closed loop: n requests, at most `conc` in flight. Messages are built before timing starts."""
    msgs = [factory() for _ in range(n)]
    sem = asyncio.Semaphore(conc)
    async def one(m):
        async with sem:
            return await timed_request(m, mm_kwargs)
    return await asyncio.gather(*[one(m) for m in msgs])

def stats(recs):
    t = [r["ttft_ms"] for r in recs if r["ttft_ms"] is not None]
    return {"n": len(t), "p50_ms": round(statistics.median(t), 1),
            "p95_ms": round(float(np.percentile(t, 95)), 1),
            "prompt_tokens": int(statistics.median(r["prompt_tokens"] for r in recs))}

async def match_text_length(target):
    """Find a word count whose text prompt has exactly `target` tokens."""
    n = max(1, target - 30)
    for _ in range(5):
        got = (await run(text_factory(n), 1, 1))[0]["prompt_tokens"]
        if got == target:
            break
        n += target - got
    return n, got

SETTINGS3 = {"full": None, "~1024": 802816, "~512": 401408, "~256": 200704}
N_WARMUP, N_RUNS, CONCS = 3, 20, [1, 8]

gate3 = []
for name, px in SETTINGS3.items():
    kw = {"max_pixels": px} if px else None
    row = {"size": name, "max_pixels": px, "raw": {}}

    await run(img_factory, N_WARMUP, 1, kw)                      # warm-up, discarded
    for c in CONCS:
        recs = await run(img_factory, N_RUNS, c, kw)
        row["raw"][f"img_c{c}"], row[f"img_c{c}"] = recs, stats(recs)

    target = row["img_c1"]["prompt_tokens"]
    n_words, got = await match_text_length(target)
    row["text_match"] = {"target": target, "got": got, "n_words": n_words}

    tf = text_factory(n_words)
    await run(tf, N_WARMUP, 1)                                    # warm-up, discarded
    for c in CONCS:
        recs = await run(tf, N_RUNS, c)
        row["raw"][f"txt_c{c}"], row[f"txt_c{c}"] = recs, stats(recs)

    row["image_share_c1"] = round(
        (row["img_c1"]["p50_ms"] - row["txt_c1"]["p50_ms"]) / row["img_c1"]["p50_ms"], 3)
    gate3.append(row)
    print(f"done: {name}")

print(f"\n{'size':>6} {'tokens':>6} {'txt tok':>7} | {'img c1 p50':>10} {'txt c1 p50':>10} {'share':>6} | "
      f"{'img c8 p50':>10} {'img c8 p95':>10} {'txt c8 p50':>10}")
for r in gate3:
    print(f"{r['size']:>6} {r['img_c1']['prompt_tokens']:>6} {r['text_match']['got']:>7} | "
          f"{r['img_c1']['p50_ms']:>10} {r['txt_c1']['p50_ms']:>10} {r['image_share_c1']:>6.1%} | "
          f"{r['img_c8']['p50_ms']:>10} {r['img_c8']['p95_ms']:>10} {r['txt_c8']['p50_ms']:>10}")

done: full
done: ~1024
done: ~512
done: ~256

  size tokens txt tok | img c1 p50 txt c1 p50  share | img c8 p50 img c8 p95 txt c8 p50
  full   1454    1454 |      451.1      181.5  59.8% |     2056.8     3494.0      626.3
 ~1024   1016    1016 |      330.2      135.3  59.0% |     1376.7     2598.8      534.8
  ~512    514     514 |      186.6       77.5  58.5% |     1074.6     1460.1      326.0
  ~256    275     275 |      126.4       51.6  59.2% |      653.8      897.6      296.3


save gate test #3

In [12]:
gate3_result = {
    "test": "gate3_speedup_and_image_share",
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "versions": VERSIONS,
    "config": {
        "server_args": SERVER_ARGS_G3,
        "image": "stop_sign 1300x876, random 8x8 corner patch per request (numpy seed 20261011), JPEG q95",
        "text_prompt": "'[NNNNNN] ' + 'word ' * n + PROMPT, length matched to image prompt_tokens",
        "prompt": PROMPT, "temperature": 0, "max_tokens": 16, "streaming": True,
        "warmup_per_point": N_WARMUP, "measured_per_point": N_RUNS, "concurrency": CONCS,
        "loop": "closed (asyncio.Semaphore)",
        "ttft_definition": "client-side, request start to first chunk with non-empty content",
        "image_share": "(img c1 p50 - same-length text c1 p50) / img c1 p50",
    },
    "rows": gate3,
}
path3 = f"{DEST}/{datetime.date.today().isoformat()}_gate3_speedup_image_share.json"
json.dump(gate3_result, open(path3, "w"), indent=2)
print("saved:", path3)
shutil.copy("/content/server_gate3.log", path3.replace("_speedup_image_share.json", "_server.log"))
print("saved:", path3.replace("_speedup_image_share.json", "_server.log"))

saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate3_speedup_image_share.json
saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate3_server.log


restart the server with default flags again for test #4

In [26]:
import re, signal

# 1. Kill every vLLM process: API server AND engine core (case-insensitive this time)
subprocess.run(["pkill", "-9", "-i", "-f", "vllm|enginecore"])

# wait until the GPU is actually free (the notebook itself holds a few hundred MiB, that's fine)
for _ in range(30):
    used = int(subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
        capture_output=True, text=True).stdout.strip())
    if used < 2000:
        break
    time.sleep(2)
print(f"GPU memory after kill: {used} MiB")
print(subprocess.run("ps aux | grep -i -E 'vllm|enginecore' | grep -v grep",
                     shell=True, capture_output=True, text=True).stdout or "no vLLM processes left")
if used >= 2000:
    raise RuntimeError("GPU still busy, see the fallback below")

# 2. Start a fresh server with default flags (prefix caching ON)
SERVER_LOG = "/content/server_default.log"
log_d = open(SERVER_LOG, "w")
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL, *SERVER_ARGS],
    stdout=log_d, stderr=subprocess.STDOUT, env=os.environ.copy(),
    start_new_session=True)

t0 = time.time()
while True:
    if server.poll() is not None:
        print("Server process died. Last log lines:")
        print(open(SERVER_LOG).read()[-3000:]); break
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print(f"Server up in {time.time() - t0:.0f}s"); break
    except requests.exceptions.RequestException:
        pass
    if time.time() - t0 > 900:
        print("Timed out."); break
    time.sleep(5)

# 3. Ask the live server which config it's actually running
m = requests.get("http://localhost:8000/metrics", timeout=5).text
print("live enable_prefix_caching:", re.search(r'enable_prefix_caching="(\w+)"', m).group(1))

GPU memory after kill: 3 MiB
no vLLM processes left
Server up in 95s
live enable_prefix_caching: True


gate #4, mixed sizes and the cache trap

In [27]:
import asyncio, random, re, datetime, shutil
from collections import Counter
import numpy as np
from openai import AsyncOpenAI

aclient = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key="none", timeout=120, max_retries=0)
PROMPT = "Describe this image in one sentence."
SIZES4 = {"full": None, "~1024": 802816, "~512": 401408, "~256": 200704}
KNOWN = {"full": 1454, "~1024": 1016, "~512": 514, "~256": 275}   # prompt tokens from gate #2 re-run
rng4 = np.random.default_rng(4)
BASE_ARR = np.array(img)

def arr_to_url(arr):
    b = io.BytesIO(); Image.fromarray(arr).save(b, format="JPEG", quality=95)
    return "data:image/jpeg;base64," + base64.b64encode(b.getvalue()).decode()

def unique_url4():
    """Stop sign with a random 8x8 corner patch: never seen before by the cache."""
    arr = BASE_ARR.copy()
    arr[:8, :8] = rng4.integers(0, 256, size=(8, 8, 3), dtype=np.uint8)
    return arr_to_url(arr)

SAME_URL4 = arr_to_url(BASE_ARR)   # the exact same image every time: the cache WILL recognise it

async def ask4(url, max_px):
    extra = {"mm_processor_kwargs": {"max_pixels": max_px}} if max_px else None
    r = await aclient.chat.completions.create(
        model=MODEL, temperature=0, max_tokens=30, extra_body=extra,
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": url}},
            {"type": "text", "text": PROMPT}]}])
    return r.usage.prompt_tokens, r.choices[0].message.content

# Phase A: correct token count per size, using fresh images (no cache involved)
expected4, phase_a = {}, []
for name, px in SIZES4.items():
    pt, ans = await ask4(unique_url4(), px)
    expected4[name] = pt
    phase_a.append({"size": name, "prompt_tokens": pt, "matches_gate2": pt == KNOWN[name], "answer": ans})
    print(f"A {name:>6}: {pt} prompt tokens (gate #2: {KNOWN[name]}) | {ans[:55]}")

# Phase B: the SAME image, sizes flipped back and forth (the direct cache trap)
order = ["full", "~256", "full", "~256", "~512", "~1024", "~256", "full"]
phase_b, ref_answers = [], {}
for name in order:
    pt, ans = await ask4(SAME_URL4, SIZES4[name])
    ref_answers.setdefault(name, ans)
    ok = pt == expected4[name]
    phase_b.append({"size": name, "prompt_tokens": pt, "ok": ok, "answer": ans})
    print(f"B {name:>6}: got {pt}, expected {expected4[name]} {'OK' if ok else '<-- MISMATCH'}")
b_bad = sum(not r["ok"] for r in phase_b)

# Phase C: 32 requests at once, random sizes, half same image / half unique
random.seed(4)
jobs = [(random.choice(list(SIZES4)), "same" if i % 2 == 0 else "unique") for i in range(32)]
urls = [SAME_URL4 if kind == "same" else unique_url4() for _, kind in jobs]
t_c = time.perf_counter()
results = await asyncio.gather(*[ask4(u, SIZES4[n]) for (n, _), u in zip(jobs, urls)],
                               return_exceptions=True)
c_errors = [repr(r) for r in results if isinstance(r, Exception)]
results = [(None, "") if isinstance(r, Exception) else r for r in results]
print(f"C finished in {time.perf_counter() - t_c:.1f}s | errors: {len(c_errors)}/32 {c_errors[:2]}")

phase_c = []
for (n, kind), (pt, ans) in zip(jobs, results):
    phase_c.append({
        "size": n, "image": kind, "prompt_tokens": pt,
        "tokens_ok": pt == expected4[n],
        "mentions_stop": "stop" in ans.lower(),
        "same_answer_as_phase_b": (ans == ref_answers[n]) if kind == "same" else None,
        "answer": ans,
    })
c_bad_tokens = sum(not r["tokens_ok"] for r in phase_c)
c_bad_answers = sum(not r["mentions_stop"] for r in phase_c)
same_rows = [r for r in phase_c if r["image"] == "same"]
c_answer_diffs = sum(not r["same_answer_as_phase_b"] for r in same_rows)

print(f"\nC size mix: {dict(Counter(n for n, _ in jobs))}")
print(f"C token mismatches: {c_bad_tokens}/32 | answers missing 'stop': {c_bad_answers}/32")
print(f"C same-image answers that differ from Phase B (info only): {c_answer_diffs}/{len(same_rows)}")

# Evidence the cache was actually on: ask the LIVE server, plus its own startup log
metrics_txt = requests.get("http://localhost:8000/metrics", timeout=5).text
cache_lines = [l for l in metrics_txt.splitlines()
               if l.startswith("vllm:") and "cache" in l]
print("\n/metrics cache lines:")
print("\n".join(cache_lines) if cache_lines else "(none found)")

log_txt = open(SERVER_LOG).read()
metric_says_on = any(re.search(r'enable_prefix_caching="?[Tt]rue', l) for l in cache_lines)
log_says_on = "enable_prefix_caching=True" in log_txt
cache_on = metric_says_on or log_says_on
print(f"\nprefix caching on -> metrics: {metric_says_on}, startup log: {log_says_on}")
print("MM cache hit rates in log:", re.findall(r"MM cache hit rate: ([\d.]+)%", log_txt)[-3:])

gate4_pass = b_bad == 0 and c_bad_tokens == 0 and c_bad_answers == 0 and cache_on
print(f"\nGATE #4: {'PASS' if gate4_pass else 'FAIL'}"
      + ("" if cache_on else "  (caching not confirmed on, so this run does not count)"))

A   full: 1454 prompt tokens (gate #2: 1454) | A stop sign is in front of a Chinese style building wit
A  ~1024: 1016 prompt tokens (gate #2: 1016) | A stop sign is in front of a Chinese style building.
A   ~512: 514 prompt tokens (gate #2: 514) | A stop sign is in front of a Chinese style archway.
A   ~256: 275 prompt tokens (gate #2: 275) | A stop sign is in front of a Chinese restaurant.
B   full: got 1454, expected 1454 OK
B   ~256: got 275, expected 275 OK
B   full: got 1454, expected 1454 OK
B   ~256: got 275, expected 275 OK
B   ~512: got 514, expected 514 OK
B  ~1024: got 1016, expected 1016 OK
B   ~256: got 275, expected 275 OK
B   full: got 1454, expected 1454 OK
C finished in 4.9s | errors: 0/32 []

C size mix: {'~1024': 9, '~512': 11, 'full': 8, '~256': 4}
C token mismatches: 0/32 | answers missing 'stop': 0/32
C same-image answers that differ from Phase B (info only): 1/16

/metrics cache lines:
vllm:kv_cache_usage_perc{engine="0",model_name="Qwen/Qwen2.5-VL-3B-Instruct"} 

save gate #4

In [28]:
from google.colab import drive
drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/visionabr_results/phase_minus1"
os.makedirs(DEST, exist_ok=True)
stamp = datetime.date.today().isoformat()

gate4 = {
    "test": "gate4_mixed_sizes_cache_trap",
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "versions": VERSIONS,
    "config": {
        "server_args": SERVER_ARGS,
        "image": "stop_sign 1300x876, JPEG q95; 'same' = unmodified image, 'unique' = random 8x8 patch (numpy seed 4)",
        "prompt": PROMPT, "temperature": 0, "max_tokens": 30, "streaming": False,
        "phase_b_order": order, "phase_c_seed": 4, "phase_c_requests": 32,
    },
    "cache_evidence": {"metrics_lines": cache_lines,
                       "metric_says_on": metric_says_on, "log_says_on": log_says_on},
    "raw": {"phase_a": phase_a, "phase_b": phase_b, "phase_c": phase_c},
    "summary": {
        "expected_prompt_tokens": expected4,
        "phase_b_mismatches": b_bad,
        "phase_c_token_mismatches": c_bad_tokens,
        "phase_c_answers_missing_stop": c_bad_answers,
        "phase_c_same_image_answer_diffs_info_only": c_answer_diffs,
        "phase_c_errors": c_errors,
        "cache_on_confirmed": cache_on,
        "pass": gate4_pass,
    },
}
path4 = f"{DEST}/{stamp}_gate4_mixed_sizes.json"
json.dump(gate4, open(path4, "w"), indent=2)
print("saved:", path4)
shutil.copy(SERVER_LOG, f"{DEST}/{stamp}_gate4_server.log")
print("saved:", f"{DEST}/{stamp}_gate4_server.log")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate4_mixed_sizes.json
saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate4_server.log


gate test #5

In [29]:
import statistics

rng5 = np.random.default_rng(5)

def unique_url5():
    arr = BASE_ARR.copy()
    arr[:8, :8] = rng5.integers(0, 256, size=(8, 8, 3), dtype=np.uint8)
    return arr_to_url(arr)

def cache_counters():
    """Read the cumulative cache counters from the live server."""
    m = requests.get("http://localhost:8000/metrics", timeout=5).text
    keys = ["vllm:prefix_cache_queries_total", "vllm:prefix_cache_hits_total",
            "vllm:mm_cache_queries_total", "vllm:mm_cache_hits_total"]
    out = {}
    for line in m.splitlines():
        for k in keys:
            if line.startswith(k + "{"):
                out[k.split(":")[1]] = float(line.rsplit(" ", 1)[1])
    return out

async def timed5(url, max_px):
    """One streaming request. TTFT = start to first chunk with text. Errors are recorded, not raised."""
    extra = {"mm_processor_kwargs": {"max_pixels": max_px}} if max_px else None
    t0 = time.perf_counter()
    try:
        stream = await aclient.chat.completions.create(
            model=MODEL, temperature=0, max_tokens=30, stream=True,
            stream_options={"include_usage": True}, extra_body=extra,
            messages=[{"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": url}},
                {"type": "text", "text": PROMPT}]}])
        ttft, ptoks = None, None
        async for chunk in stream:
            if ttft is None and chunk.choices and chunk.choices[0].delta.content:
                ttft = (time.perf_counter() - t0) * 1000
            if chunk.usage:
                ptoks = chunk.usage.prompt_tokens
        return {"ttft_ms": ttft, "prompt_tokens": ptoks,
                "total_ms": (time.perf_counter() - t0) * 1000, "error": None}
    except Exception as e:
        return {"ttft_ms": None, "prompt_tokens": None,
                "total_ms": (time.perf_counter() - t0) * 1000, "error": repr(e)}

async def flood5(max_px, conc, n):
    """Closed loop: n unique-image requests, at most `conc` in flight. Images built before timing."""
    urls = [unique_url5() for _ in range(n)]
    sem = asyncio.Semaphore(conc)
    async def one(u):
        async with sem:
            return await timed5(u, max_px)
    t0 = time.perf_counter()
    recs = await asyncio.gather(*[one(u) for u in urls])
    wall = time.perf_counter() - t0
    t = [r["ttft_ms"] for r in recs if r["ttft_ms"] is not None]
    summ = {"p50_ms": round(statistics.median(t)), "p95_ms": round(float(np.percentile(t, 95))),
            "req_per_s": round(n / wall, 2), "ok": len(t),
            "errors": sum(r["error"] is not None for r in recs), "wall_s": round(wall, 1)}
    return recs, summ

LEVELS5 = {"full": None, "~512": 401408, "~256": 200704}
OCT3 = {(16, "full"): (1809, 6808, 2.03), (16, "~512"): (1118, 2601, 5.35), (16, "~256"): (1035, 1466, 8.15),
        (32, "full"): (3769, 14170, 2.27), (32, "~512"): (1547, 5202, 6.95), (32, "~256"): (1221, 2749, 10.84)}
N5 = 64

cache_before = cache_counters()
gate5 = []
for conc in [16, 32]:
    for name, px in LEVELS5.items():
        await flood5(px, 4, 4)                                   # warm-up, discarded
        recs, summ = await flood5(px, conc, N5)
        o = OCT3[(conc, name)]
        gate5.append({"concurrency": conc, "size": name, "max_pixels": px, "summary": summ,
                      "oct3": {"p50_ms": o[0], "p95_ms": o[1], "req_per_s": o[2]}, "records": recs})
        print(f"c={conc:>2} {name:>5}: p50 {summ['p50_ms']:>5} (Oct3 {o[0]:>5}) | "
              f"p95 {summ['p95_ms']:>5} (Oct3 {o[1]:>5}) | {summ['req_per_s']:>5} req/s (Oct3 {o[2]:>5}) | "
              f"ok {summ['ok']}/{N5} err {summ['errors']}")
cache_after = cache_counters()
cache_delta = {k: cache_after[k] - cache_before.get(k, 0) for k in cache_after}
print("\ncache counter change during #5:", cache_delta)
print("image cache hits (should be 0, every image unique):", cache_delta.get("mm_cache_hits_total"))

c=16  full: p50  1979 (Oct3  1809) | p95  6141 (Oct3  6808) |  2.19 req/s (Oct3  2.03) | ok 64/64 err 0
c=16  ~512: p50   950 (Oct3  1118) | p95  2434 (Oct3  2601) |  5.71 req/s (Oct3  5.35) | ok 64/64 err 0
c=16  ~256: p50   512 (Oct3  1035) | p95  1374 (Oct3  1466) |   9.5 req/s (Oct3  8.15) | ok 64/64 err 0
c=32  full: p50  3691 (Oct3  3769) | p95 13876 (Oct3 14170) |  2.15 req/s (Oct3  2.27) | ok 64/64 err 0
c=32  ~512: p50  1761 (Oct3  1547) | p95  5118 (Oct3  5202) |  5.84 req/s (Oct3  6.95) | ok 64/64 err 0
c=32  ~256: p50  1647 (Oct3  1221) | p95  2713 (Oct3  2749) |  9.88 req/s (Oct3 10.84) | ok 64/64 err 0

cache counter change during #5: {'prefix_cache_queries_total': 305048.0, 'prefix_cache_hits_total': 0.0, 'mm_cache_queries_total': 408.0, 'mm_cache_hits_total': 0.0}
image cache hits (should be 0, every image unique): 0.0


save gate #5

In [30]:
gate5_result = {
    "test": "gate5_overload_closed_loop",
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "versions": VERSIONS,
    "config": {
        "server_args": SERVER_ARGS, "prefix_caching": "on (default, same as Oct 3)",
        "image": "stop_sign 1300x876, random 8x8 corner patch per request (numpy seed 5), JPEG q95, no repeats",
        "prompt": PROMPT, "temperature": 0, "max_tokens": 30, "streaming": True,
        "requests_per_row": N5, "warmup": "4 requests at concurrency 4 per row, discarded",
        "row_order": "c16 full, c16 ~512, c16 ~256, c32 full, c32 ~512, c32 ~256 (one server, no restart)",
        "loop": "closed (asyncio.Semaphore)", "images_built_before_timing": True,
        "p50": "statistics.median", "p95": "numpy.percentile linear",
        "throughput": "requests / wall time of the measured batch",
    },
    "cache_counter_delta": cache_delta,
    "rows": gate5,
}
path5 = f"{DEST}/{stamp}_gate5_overload.json"
json.dump(gate5_result, open(path5, "w"), indent=2)
print("saved:", path5)

saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate5_overload.json


gate test #6

In [31]:
rng6 = np.random.default_rng(6)

def unique_url6():
    arr = BASE_ARR.copy()
    arr[:8, :8] = rng6.integers(0, 256, size=(8, 8, 3), dtype=np.uint8)
    return arr_to_url(arr)

LOAD_KEYS = ("vllm:num_requests_running", "vllm:num_requests_waiting", "vllm:kv_cache_usage_perc")

def load_metrics6():
    """One /metrics read. Returns (values, how long the read took in ms)."""
    t = time.perf_counter()
    txt = requests.get("http://localhost:8000/metrics", timeout=5).text
    dt = (time.perf_counter() - t) * 1000
    vals = {}
    for line in txt.splitlines():
        for k in LOAD_KEYS:
            if line.startswith(k + "{"):          # exact name, skips e.g. num_requests_waiting_by_reason
                vals[k.split(":")[1]] = float(line.rsplit(" ", 1)[1])
    return vals, dt

# Which preemption metrics does vLLM 0.30 expose?
m_all = requests.get("http://localhost:8000/metrics", timeout=5).text
preempt_lines = [l for l in m_all.splitlines() if "preempt" in l.lower() and not l.startswith("#")]
print("preemption metrics:", preempt_lines or "(none)")

series = []
async def poll6(stop, t0):
    while not stop.is_set():
        vals, dt = await asyncio.to_thread(load_metrics6)
        series.append({"t_s": round(time.perf_counter() - t0, 3), "poll_ms": round(dt, 1), **vals})
        await asyncio.sleep(0.25)

urls6 = [unique_url6() for _ in range(48)]           # built before the burst
idle, _ = load_metrics6()
print("idle:", idle)

t0 = time.perf_counter()
stop = asyncio.Event()
poller = asyncio.create_task(poll6(stop, t0))
await asyncio.sleep(1.0)                              # 1 s of calm baseline
t_burst = round(time.perf_counter() - t0, 3)
recs6 = await asyncio.gather(*[timed5(u, None) for u in urls6])   # all 48 at once, no semaphore
t_done = round(time.perf_counter() - t0, 3)
await asyncio.sleep(1.5)                              # watch it drain back to idle
stop.set(); await poller

for s_ in series:
    s_["in_system"] = s_.get("num_requests_running", 0) + s_.get("num_requests_waiting", 0)

peak = {
    "running": max(s_["num_requests_running"] for s_ in series),
    "waiting": max(s_["num_requests_waiting"] for s_ in series),
    "in_system_same_moment": max(s_["in_system"] for s_ in series),
    "kv_cache_usage_perc": max(s_["kv_cache_usage_perc"] for s_ in series),
}
polls = [s_["poll_ms"] for s_ in series]
ttfts6 = [r["ttft_ms"] for r in recs6 if r["ttft_ms"] is not None]

print(f"\nburst started at t={t_burst}s, all 48 done at t={t_done}s, errors {sum(r['error'] is not None for r in recs6)}")
print("peaks:", peak)
print(f"/metrics read time: p50 {statistics.median(polls):.1f} ms, max {max(polls):.1f} ms, {len(polls)} samples")
print(f"burst TTFT: p50 {statistics.median(ttfts6):.0f} ms, max {max(ttfts6):.0f} ms")
print("\ntimeline (every 2nd sample):")
print(f"{'t_s':>6} {'run':>4} {'wait':>4} {'total':>5} {'kv%':>5}")
for s_ in series[::2]:
    print(f"{s_['t_s']:>6} {s_['num_requests_running']:>4.0f} {s_['num_requests_waiting']:>4.0f} "
          f"{s_['in_system']:>5.0f} {100 * s_['kv_cache_usage_perc']:>5.1f}")

preemption metrics: ['vllm:num_preemptions_total{engine="0",model_name="Qwen/Qwen2.5-VL-3B-Instruct"} 0.0', 'vllm:num_preemptions_created{engine="0",model_name="Qwen/Qwen2.5-VL-3B-Instruct"} 1.7917556428595216e+09', 'vllm:request_num_preemptions_bucket{engine="0",le="1.0",model_name="Qwen/Qwen2.5-VL-3B-Instruct"} 452.0', 'vllm:request_num_preemptions_bucket{engine="0",le="2.0",model_name="Qwen/Qwen2.5-VL-3B-Instruct"} 452.0', 'vllm:request_num_preemptions_bucket{engine="0",le="3.0",model_name="Qwen/Qwen2.5-VL-3B-Instruct"} 452.0', 'vllm:request_num_preemptions_bucket{engine="0",le="4.0",model_name="Qwen/Qwen2.5-VL-3B-Instruct"} 452.0', 'vllm:request_num_preemptions_bucket{engine="0",le="5.0",model_name="Qwen/Qwen2.5-VL-3B-Instruct"} 452.0', 'vllm:request_num_preemptions_bucket{engine="0",le="10.0",model_name="Qwen/Qwen2.5-VL-3B-Instruct"} 452.0', 'vllm:request_num_preemptions_bucket{engine="0",le="20.0",model_name="Qwen/Qwen2.5-VL-3B-Instruct"} 452.0', 'vllm:request_num_preemptions_buc

save gate #6

In [32]:
gate6 = {
    "test": "gate6_load_signal",
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "versions": VERSIONS,
    "config": {
        "server_args": SERVER_ARGS,
        "burst": "48 full-size unique images (random patch, numpy seed 6), all sent at once, streaming, max_tokens 30",
        "poll_interval_s": 0.25, "baseline_s": 1.0, "drain_s": 1.5,
        "metrics": list(LOAD_KEYS),
    },
    "idle": idle,
    "preemption_metric_lines": preempt_lines,
    "summary": {
        "peaks": peak, "t_burst_s": t_burst, "t_done_s": t_done,
        "poll_ms_p50": round(statistics.median(polls), 1), "poll_ms_max": round(max(polls), 1),
        "burst_ttft_p50_ms": round(statistics.median(ttfts6)), "burst_ttft_max_ms": round(max(ttfts6)),
        "errors": sum(r["error"] is not None for r in recs6),
    },
    "series": series,
    "records": recs6,
}
path6 = f"{DEST}/{stamp}_gate6_load_signal.json"
json.dump(gate6, open(path6, "w"), indent=2)
print("saved:", path6)

saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate6_load_signal.json


load the datasets

In [33]:
!pip install -q datasets
from datasets import load_dataset

SPECS7 = {
    "photos (POPE)":    dict(path="lmms-lab/POPE", name=None, split="test",
                             suffix=" Answer yes or no."),
    "charts (ChartQA)": dict(path="lmms-lab/ChartQA", name=None, split="test",
                             suffix=" Answer with a single word or number."),
    "text (TextVQA)":   dict(path="lmms-lab/textvqa", name=None, split="validation",
                             suffix=" Answer with a single word or short phrase."),
    "docs (DocVQA)":    dict(path="lmms-lab/DocVQA", name="DocVQA", split="validation",
                             suffix=" Answer with a single word or short phrase."),
}
N_PER = 200
META_KEYS = ("question_id", "questionId", "image_id", "docId", "category", "type", "image_source")

samples7 = {}
for kind, s in SPECS7.items():
    ds = load_dataset(s["path"], s["name"], split=s["split"], streaming=True)
    rows = []
    for i, ex in enumerate(ds):                       # first 200 rows, same as Oct 3
        im = ex["image"].convert("RGB")
        rows.append({
            "idx": i,
            "url": arr_to_url(np.array(im)),
            "q": ex["question"] + s["suffix"],
            "golds": ex.get("answers") or [ex.get("answer")],
            "wh": list(im.size),
            "meta": {k: ex[k] for k in META_KEYS if ex.get(k) is not None},
        })
        if len(rows) >= N_PER:
            break
    samples7[kind] = rows
    ws, hs = [r["wh"][0] for r in rows], [r["wh"][1] for r in rows]
    print(f"{kind:<17} loaded {len(rows)} | median size {int(np.median(ws))}x{int(np.median(hs))}")

print("\nPOPE category mix:", dict(Counter(r["meta"].get("category") for r in samples7["photos (POPE)"])))

README.md:   0%|          | 0.00/2.16k [00:00<?, ?B/s]

photos (POPE)     loaded 200 | median size 640x427


README.md:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

charts (ChartQA)  loaded 200 | median size 340x415


README.md:   0%|          | 0.00/2.00k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

text (TextVQA)    loaded 200 | median size 1024x768


README.md:   0%|          | 0.00/2.69k [00:00<?, ?B/s]

docs (DocVQA)     loaded 200 | median size 1701x2213

POPE category mix: {'adversarial': 200}


gate test #7

In [34]:
import math

SIZES7 = {"full": None, "~1024": 802816, "~512": 401408, "~256": 200704}
OCT3_7 = {
    "photos (POPE)":    {"full": 84.5, "~1024": 84.5, "~512": 84.5, "~256": 85.0},
    "charts (ChartQA)": {"full": 61.0, "~1024": 61.5, "~512": 62.5, "~256": 59.5},
    "text (TextVQA)":   {"full": 85.5, "~1024": 84.5, "~512": 81.0, "~256": 75.0},
    "docs (DocVQA)":    {"full": 90.5, "~1024": 90.5, "~512": 83.5, "~256": 58.0},
}

def norm(s):
    return re.sub(r"[^\w\s.]", "", str(s).lower()).strip().rstrip(".")

def correct(kind, pred, golds):
    """Same scoring as Oct 3."""
    p = norm(pred)
    if kind.startswith("photos"):
        return p.startswith(norm(golds[0]))
    if kind.startswith("charts"):                     # relaxed: numbers within 5%
        try:
            g = float(norm(golds[0]).replace(",", ""))
            return abs(float(p.replace(",", "")) - g) <= 0.05 * abs(g)
        except ValueError:
            return p == norm(golds[0])
    return any(p == norm(g) for g in golds)

def predicted_image_tokens(w, h, max_pixels=None, f=28, min_pixels=3136, default_max=12845056):
    """Qwen2.5-VL resize rule (checked on the stop sign in gate #2), plus 2 marker tokens."""
    max_px = max_pixels or default_max
    h_bar, w_bar = max(f, round(h / f) * f), max(f, round(w / f) * f)
    if h_bar * w_bar > max_px:
        beta = math.sqrt(h * w / max_px)
        h_bar, w_bar = math.floor(h / beta / f) * f, math.floor(w / beta / f) * f
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (h * w))
        h_bar, w_bar = math.ceil(h * beta / f) * f, math.ceil(w * beta / f) * f
    return (h_bar // f) * (w_bar // f) + 2

sem7 = asyncio.Semaphore(16)

async def ask7(content, max_px=None):
    """One request (image+text or text only). Errors recorded, not raised."""
    extra = {"mm_processor_kwargs": {"max_pixels": max_px}} if max_px else None
    async with sem7:
        try:
            r = await aclient.chat.completions.create(
                model=MODEL, temperature=0, max_tokens=20, extra_body=extra,
                messages=[{"role": "user", "content": content}])
            return {"pred": r.choices[0].message.content or "",
                    "prompt_tokens": r.usage.prompt_tokens, "error": None}
        except Exception as e:
            return {"pred": "", "prompt_tokens": None, "error": repr(e)}

t7 = time.perf_counter()
for kind, rows in samples7.items():
    # text-only baseline per question, to get real image tokens by subtraction (like gate #1)
    base = await asyncio.gather(*[ask7(r["q"]) for r in rows])
    for r, b in zip(rows, base):
        r["text_tokens"] = b["prompt_tokens"]
        r["res"] = {}
    for size, px in SIZES7.items():
        outs = await asyncio.gather(*[ask7([
            {"type": "image_url", "image_url": {"url": r["url"]}},
            {"type": "text", "text": r["q"]}], px) for r in rows])
        for r, o in zip(rows, outs):
            img_tok = (o["prompt_tokens"] - r["text_tokens"]
                       if o["prompt_tokens"] is not None and r["text_tokens"] is not None else None)
            r["res"][size] = {**o, "image_tokens": img_tok,
                              "rule_tokens": predicted_image_tokens(*r["wh"], px),
                              "correct": correct(kind, o["pred"], r["golds"])}
    print(f"done: {kind} ({time.perf_counter() - t7:.0f}s)")

print(f"\n{'type':<17} | " + " | ".join(f"{s:>22}" for s in SIZES7))
summary7 = {}
for kind, rows in samples7.items():
    summary7[kind] = {}
    cells = []
    for size in SIZES7:
        acc = 100 * sum(r["res"][size]["correct"] for r in rows) / len(rows)
        lost = sum(r["res"]["full"]["correct"] and not r["res"][size]["correct"] for r in rows)
        gained = sum(r["res"][size]["correct"] and not r["res"]["full"]["correct"] for r in rows)
        toks = [r["res"][size]["image_tokens"] for r in rows if r["res"][size]["image_tokens"] is not None]
        rule_ok = sum(r["res"][size]["image_tokens"] == r["res"][size]["rule_tokens"] for r in rows)
        errs = sum(r["res"][size]["error"] is not None for r in rows)
        summary7[kind][size] = {"acc": round(acc, 1), "lost": lost, "gained": gained,
                                "oct3_acc": OCT3_7[kind][size],
                                "median_image_tokens": int(np.median(toks)) if toks else None,
                                "rule_matches": rule_ok, "errors": errs}
        pg = "" if size == "full" else f" -{lost}/+{gained}"
        cells.append(f"{acc:5.1f} ({OCT3_7[kind][size]:4.1f}){pg:>9}")
    print(f"{kind:<17} | " + " | ".join(f"{c:>22}" for c in cells))

print("\nmedian real image tokens (incl. 2 markers):")
for kind in samples7:
    print(f"  {kind:<17} " + "  ".join(f"{s}: {summary7[kind][s]['median_image_tokens']}" for s in SIZES7))
print("\nresize rule matched / errors (out of 200):")
for kind in samples7:
    print(f"  {kind:<17} " + "  ".join(
        f"{s}: {summary7[kind][s]['rule_matches']}/{summary7[kind][s]['errors']}" for s in SIZES7))

done: photos (POPE) (23s)
done: charts (ChartQA) (62s)
done: text (TextVQA) (177s)
done: docs (DocVQA) (330s)

type              |                   full |                  ~1024 |                   ~512 |                   ~256
photos (POPE)     |   84.5 (84.5)          |   84.5 (84.5)    -0/+0 |   84.5 (84.5)    -0/+0 |   85.0 (85.0)    -5/+6
charts (ChartQA)  |   62.0 (61.0)          |   61.0 (61.5)    -2/+0 |   63.0 (62.5)    -2/+4 |   60.5 (59.5)    -5/+2
text (TextVQA)    |   85.5 (85.5)          |   84.5 (84.5)    -2/+0 |   80.0 (81.0)   -17/+6 |   74.5 (75.0)   -27/+5
docs (DocVQA)     |   90.5 (90.5)          |   90.5 (90.5)    -5/+5 |   84.0 (83.5)   -18/+5 |   57.5 (58.0)   -67/+1

median real image tokens (incl. 2 markers):
  photos (POPE)     full: 347  ~1024: 347  ~512: 347  ~256: 236
  charts (ChartQA)  full: 182  ~1024: 182  ~512: 182  ~256: 182
  text (TextVQA)    full: 1001  ~1024: 1001  ~512: 496  ~256: 236
  docs (DocVQA)     full: 5014  ~1024: 1001  ~512: 496  ~256

save gate #7

In [35]:
records7 = []
for kind, rows in samples7.items():
    for r in rows:
        records7.append({"type": kind, "idx": r["idx"], "meta": r["meta"], "wh": r["wh"],
                         "question": r["q"], "golds": r["golds"], "text_tokens": r["text_tokens"],
                         "by_size": r["res"]})

gate7 = {
    "test": "gate7_accuracy_by_type",
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "versions": VERSIONS,
    "config": {
        "server_args": SERVER_ARGS,
        "datasets": {k: {kk: v[kk] for kk in ("path", "name", "split", "suffix")} for k, v in SPECS7.items()},
        "sampling": "first 200 rows of each split, streaming, unshuffled (same as Oct 3)",
        "image_encoding": "dataset image -> RGB -> JPEG q95 -> base64",
        "temperature": 0, "max_tokens": 20, "concurrency": 16,
        "scoring": "POPE startswith gold; ChartQA relaxed 5% numeric else exact; TextVQA/DocVQA exact match to any gold after normalisation",
        "image_tokens": "prompt_tokens(image+question) - prompt_tokens(question only), incl. 2 markers",
    },
    "summary": summary7,
    "records": records7,
}
path7 = f"{DEST}/{stamp}_gate7_accuracy.json"
json.dump(gate7, open(path7, "w"), indent=2)
print("saved:", path7, f"({os.path.getsize(path7) / 1e6:.1f} MB)")

saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate7_accuracy.json (1.0 MB)


how many answers flip when the input is identical?

In [36]:
print("per size: images that actually shrank | right/wrong flips on images that did NOT change")
for kind, rows in samples7.items():
    parts = []
    for size in ["~1024", "~512", "~256"]:
        same = [r for r in rows if r["res"][size]["image_tokens"] == r["res"]["full"]["image_tokens"]]
        shrank = len(rows) - len(same)
        flips = sum(r["res"][size]["correct"] != r["res"]["full"]["correct"] for r in same)
        parts.append(f"{size}: {shrank:>3} shrank, {flips} flips on {len(same)} unchanged")
    print(f"{kind:<17} " + " | ".join(parts))

per size: images that actually shrank | right/wrong flips on images that did NOT change
photos (POPE)     ~1024:   0 shrank, 0 flips on 200 unchanged | ~512:   6 shrank, 0 flips on 194 unchanged | ~256: 140 shrank, 0 flips on 60 unchanged
charts (ChartQA)  ~1024:   0 shrank, 2 flips on 200 unchanged | ~512:  54 shrank, 3 flips on 146 unchanged | ~256:  60 shrank, 1 flips on 140 unchanged
text (TextVQA)    ~1024:  29 shrank, 1 flips on 171 unchanged | ~512: 200 shrank, 0 flips on 0 unchanged | ~256: 200 shrank, 0 flips on 0 unchanged
docs (DocVQA)     ~1024: 200 shrank, 0 flips on 0 unchanged | ~512: 200 shrank, 0 flips on 0 unchanged | ~256: 200 shrank, 0 flips on 0 unchanged
